# Malay Rojak Lemmatiser — training on a Colab T4

Trains both adapters (Llama 3.2 3B and Qwen 2.5 3B) on the full 7,200-row
training set. On a T4 this is roughly 4-7x the throughput of a 4 GB laptop
card, which turns a 12.8-hour epoch into something that fits in a session.

**Before you start:** `Runtime` -> `Change runtime type` -> **T4 GPU** -> Save.

**Run the cells in order.** Cell 5 is a 2-minute smoke test — do not skip it,
it catches a broken install or missing data before you commit hours to a run.

## 1. Confirm you actually got a T4

Colab hands out whatever is free. If this prints `CPU` or errors, go back to
`Runtime` -> `Change runtime type`. If it gives you an L4 or A100, even better —
nothing below needs changing.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 2. Install Unsloth

Takes 2-3 minutes. Ignore the pip dependency-resolver warnings; Colab always
prints them and they do not affect training.

In [ ]:
!pip install --quiet unsloth
print('installed')


## 3. Mount Google Drive

This is where the trained adapters get saved. It matters: Colab's own disk is
wiped when the session ends or disconnects, and a free session can drop at any
time. Writing the adapters to Drive means a disconnect costs you the run in
progress, not the run you already finished.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/lemmatizer_adapters'
os.makedirs(SAVE_DIR, exist_ok=True)
print('adapters will be saved to:', SAVE_DIR)


## 4. Upload the code and data

Run this cell, click **Choose Files**, and pick `colab_bundle.zip` from your
project folder. It is 1.1 MB — a few seconds.

The bundle holds exactly four files: `train.py`, `skill.py`, and the two
`*_reasoning.jsonl` splits. Nothing else is needed, because `train.py` reads its
system prompt from `skill.py` and its data from those two files.

In [ ]:
import zipfile, os
from google.colab import files

os.makedirs('/content/lemmatizer', exist_ok=True)
%cd /content/lemmatizer

up = files.upload()                     # pick colab_bundle.zip
for name in up:
    with zipfile.ZipFile(name) as z:
        z.extractall('/content/lemmatizer')
    print('extracted', name)

# Verify the four files arrived and the row counts are what we expect.
for p, want in (('train.py', None), ('skill.py', None),
                ('unsloth/data/train_reasoning.jsonl', 7200),
                ('unsloth/data/dev_reasoning.jsonl', 800)):
    n = sum(1 for _ in open(p, encoding='utf-8')) if want else None
    print('%-40s %s' % (p, ('%d rows' % n) if want else 'ok'))
    assert want is None or n == want, 'expected %d rows, got %d' % (want, n)


## 5. Smoke test — 20 steps, about 2 minutes

Do not skip this. It proves four things before you spend hours:

- the install works and the GPU is visible
- the data loaded and tokenises
- **the loss mask lands on the answer and not the prompt** (`--verify` checks this
  against a real collated batch; a wrong mask trains a model that recites the
  system prompt back at you, and it fails silently otherwise)
- the loss is actually falling

Watch the `it/s` in the progress bar — that is your real throughput, and it tells
you what the full run below will cost.

In [ ]:
!python train.py --model llama --bs 4 --accum 1 --max-steps 20 --evals 1


## 6. Train Llama — the full epoch

`--budget-mins 0` removes the wall-clock cap, so the run takes the full epoch:
7,200 rows at effective batch 4 = **1,800 steps**.

`--bs 4 --accum 1` is the same optimisation as the laptop's `1 x 4` — identical
effective batch, identical gradients — but a per-device batch of 4 keeps a T4
properly fed, where a batch of 1 leaves it waiting on kernel launches.

Expect roughly 1.5-3 hours. **Keep this browser tab open**; a free Colab session
disconnects when it decides you are idle.

In [ ]:
!python train.py --model llama --bs 4 --accum 1 --budget-mins 0


## 7. Save Llama to Drive — do this immediately

Before starting the second run, not after. If Qwen's run dies halfway it must
not take Llama's finished adapter with it.

In [ ]:
import shutil, os, json

def save(name):
    src = '/content/lemmatizer/unsloth/lemmatizer_lora_%s' % name
    dst = os.path.join(SAVE_DIR, 'lemmatizer_lora_%s' % name)
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(src, dst)
    # The loss curve lives in the checkpoints, not the adapter, and
    # plot_training.py reads it back on your laptop.
    runs = '/content/lemmatizer/unsloth/training_runs/lemmatizer_lora_%s' % name
    if os.path.isdir(runs):
        for ck in sorted(os.listdir(runs)):
            st = os.path.join(runs, ck, 'trainer_state.json')
            if os.path.exists(st):
                out = os.path.join(SAVE_DIR, 'training_runs',
                                   'lemmatizer_lora_%s' % name, ck)
                os.makedirs(out, exist_ok=True)
                shutil.copy2(st, out)
    print('saved ->', dst)
    print(open(os.path.join(dst, 'run_meta.json'), encoding='utf-8').read())

save('llama')


## 8. Train Qwen — same length, automatically

With `--budget-mins 0` both models run the full epoch, so both land on 1,800
steps by construction — no `--max-steps` needed. The script re-reads Llama's
`run_meta.json` and prints `[fairness] matches` to confirm it.

That matching matters: step count *is* training-data volume, so unequal steps
would mean the comparison measured which model trained longer rather than which
pretraining suits Malay.

If the session dropped after cell 7, you can start a fresh session, re-run cells
1-5, and come straight here — Llama is safe in Drive. In that case pass
`--max-steps 1800` explicitly, since `run_meta.json` for Llama will no longer be
on the local disk for the fairness check to find.

In [ ]:
!python train.py --model qwen --bs 4 --accum 1 --budget-mins 0


## 9. Save Qwen to Drive

In [ ]:
save('qwen')


## 10. Bring the adapters home

They are already in your Drive at `MyDrive/lemmatizer_adapters/`, so the
simplest route is to download the folder from drive.google.com and drop the two
`lemmatizer_lora_*` directories into your project's `unsloth/` folder, replacing
the laptop-trained ones.

The cell below zips them for a direct browser download instead — about 200 MB,
so Drive is usually less painful.

In [ ]:
import shutil
shutil.make_archive('/content/adapters', 'zip', SAVE_DIR)
print('%.0f MB' % (os.path.getsize('/content/adapters.zip') / 1e6))

from google.colab import files
files.download('/content/adapters.zip')


## Back on your laptop

Put the two adapter folders into `unsloth/`, then:

```bash
# loss curves for both, from the trainer_state.json files you copied back
python plot_training.py

# sanity-check the words that failed before
python main.py --model llama-ft "aq nk mandi"

# then the real evaluation
python crawl/evaluate.py --model --adapter lemmatizer_lora_llama
```

`aq -> aku` appeared in 40 of the 7,200 rows. The laptop run sampled ~934 rows and
so was expected to see it about 5 times; a full epoch sees all 40. That is the
specific reason this run should fix it.